# Practice 3 (Part 1): Sentiment Analysis with Hugging Face Pre-trained Models

## Pre-trained Pipeline Exploration & Deep-Dive into Tokenizer & PyTorch Tensors
This notebook guides through high-level and low-level NLP sentiment analysis using pre-trained Transformer models from the Hugging Face Hub. It fulfills Exercise 1 of Lab 3 by examining the inference pipeline, subword tokenization, tensor representations, and manual forward pass with PyTorch.

## Roadmap Table
| Step | Description | What it does | Import path |
|---|---|---|---|
| 1 | Environment & Seed Setup | Configures dynamic device and seed for reproducibility | [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md) |
| 2 | High-Level Inference Pipeline | Runs sentiment analysis using pre-trained pipeline | `transformers.pipeline` |
| 3 | Tokenizer Deep-Dive | Inspects WordPiece subwords, token IDs, and special tokens | `transformers.AutoTokenizer` |
| 4 | Low-Level PyTorch Forward Pass | Feeds tensors into model and computes Softmax probabilities | `transformers.AutoModelForSequenceClassification` |
| 5 | Batch Processing & Padding | Demonstrates attention masks and pad token suppression | `torch.nn.functional` |
| 6 | Linguistic Stress-Testing | Evaluates model robustness on complex negation and idioms | [`PIPELINE_EXPLANATION.md`](../../../docs/lab3/PIPELINE_EXPLANATION.md) |
---

## References
- Task Brief: [`labs/lab3/QUESTIOn.md`](../QUESTIOn.md)
- Pipeline Architecture: [`docs/lab3/PIPELINE_EXPLANATION.md`](../../../docs/lab3/PIPELINE_EXPLANATION.md)
- Pre-trained Model: [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english)
- Framework Governance: [`agents/rules/PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md)
- Notebook Standards: [`agents/rules/NOTEBOOK_HEADER_CONVENTION.md`](../../../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md)


## 1. Environment & Reproducibility Setup

Tuân thủ quy tắc [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md), ta thiết lập cấu hình thiết bị động (`device agnostic`) và cố định seed ngẫu nhiên nhằm đảm bảo tính tái lập (reproducibility).

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Active Device: {device}")
if torch.cuda.is_available():
    print(f"[Setup] GPU Name: {torch.cuda.get_device_name(0)}")


## 2. High-Level Inference với Hugging Face `pipeline`

Hàm `pipeline()` trong Hugging Face là tầng trừu tượng hóa cao nhất (high-level API), đóng gói toàn bộ quy trình từ: Tiền xử lý (Tokenization) $\rightarrow$ Đưa qua mô hình (Model Forward) $\rightarrow$ Hậu xử lý (Softmax & ánh xạ nhãn).

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Khởi tạo pipeline phân tích cảm xúc (sentiment-analysis)
device_idx = 0 if torch.cuda.is_available() else -1
classifier = pipeline("sentiment-analysis", model=model_name, device=device_idx)

sample_sentences = [
    "Just have trying, don't end up like me",
    "I love this movie, the plot is brilliant and moving!",
    "A complete waste of time and money."
]

results = classifier(sample_sentences)
for sent, res in zip(sample_sentences, results):
    print(f"Sentence: \"{sent}\"")
    print(f"Prediction -> Label: {res['label']}, Confidence: {res['score']:.4f}\n")


## 3. Mổ xẻ Chi tiết Tokenizer (WordPiece, Tokens & IDs)

Tokenizer trong mô hình Transformer có nhiệm vụ chuyển đổi văn bản ký tự thành các mã chỉ mục số nguyên (Token IDs). `DistilBERT` sử dụng thuật toán **WordPiece** với từ điển kích thước 30,522 tokens.

Các token đặc biệt quan trọng:
- `[CLS]` (ID: `101`): Đặt ở đầu câu, biểu diễn tổng hợp của toàn bộ câu cho tác vụ phân loại.
- `[SEP]` (ID: `102`): Đặt ở cuối câu để báo hiệu kết thúc câu.
- `[PAD]` (ID: `0`): Dùng để đệm độ dài các câu trong batch cho bằng nhau.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

test_sentence = sample_sentences[0]  # "Just have trying, don't end up like me"
tokens = tokenizer.tokenize(test_sentence)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Văn bản gốc: \"{test_sentence}\"")
print(f"Subword Tokens : {tokens}")
print(f"Token IDs      : {token_ids}")

# Quan sát mã hóa đầy đủ kèm special tokens
full_encoding = tokenizer(test_sentence)
print(f"\nFull Input IDs (có [CLS] và [SEP]): {full_encoding['input_ids']}")
print(f"Decoded Tokens : {tokenizer.convert_ids_to_tokens(full_encoding['input_ids'])}")
print(f"Attention Mask : {full_encoding['attention_mask']}")


## 4. Suy luận Tường minh Cấp thấp với PyTorch (Low-Level Forward Pass)

Để hiểu bản chất toán học đằng sau `pipeline`, ta thực hiện suy luận native bằng PyTorch:
1. Mã hóa văn bản thành tensor (`return_tensors="pt"`).
2. Truyền tensor qua mô hình trong khối `torch.no_grad()` để tắt tính gradient, tiết kiệm bộ nhớ.
3. Nhận về ma trận **Logits** chưa chuẩn hóa.
4. Áp dụng hàm **Softmax** để chuyển logits thành phân phối xác suất ($P \in [0, 1]$).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

inputs = tokenizer(
    sample_sentences,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
).to(device)

print("Tensor input_ids shape:", inputs["input_ids"].shape)
print("Tensor attention_mask shape:", inputs["attention_mask"].shape)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=1)
    predicted_classes = torch.argmax(probabilities, dim=1)

print("\n--- Logits (Raw Unnormalized Outputs) ---")
print(logits.cpu().numpy())

print("\n--- Softmax Probabilities (Negative, Positive) ---")
print(probabilities.cpu().numpy())

for idx, (sent, pred, prob) in enumerate(zip(sample_sentences, predicted_classes, probabilities)):
    label_str = model.config.id2label[pred.item()]
    confidence = prob[pred].item()
    print(f"[{idx+1}] \"{sent}\" -> {label_str} ({confidence*100:.2f}%)")


## 5. Vai trò của `attention_mask` trong Batch Processing

Khi nạp nhiều câu có độ dài khác nhau vào cùng một batch, các câu ngắn bắt buộc phải được đệm (padding) bằng token `[PAD]` (ID 0). `attention_mask` chứa giá trị $0$ tại các vị trí đệm để tầng Self-Attention hoàn toàn bỏ qua chúng khi tính tích vô hướng Query-Key.

In [ ]:
print("Ma trận attention_mask của batch:")
print(inputs["attention_mask"].cpu().numpy())

print("\nQuan sát câu ngắn nhất: có các số 0 ở cuối đại diện cho token [PAD]")
for i, mask in enumerate(inputs["attention_mask"]):
    num_real_tokens = mask.sum().item()
    num_pad_tokens = len(mask) - num_real_tokens
    print(f"Câu {i+1}: {num_real_tokens} real tokens | {num_pad_tokens} pad tokens")


## 6. Kiểm thử Khả năng Xử lý Cấu trúc Ngôn ngữ Phức hợp (Linguistic Stress-Testing)

Các mô hình truyền thống (BoW, Naive Bayes) thường thất bại khi gặp các cấu trúc phủ định đảo nghĩa (ví dụ: *"not bad"* chứa từ *bad* nhưng mang nghĩa tích cực). Ta kiểm thử năng lực của cơ chế Self-Attention trên các trường hợp biên này.

In [ ]:
edge_cases = [
    "It is not bad at all.",
    "I expected it to be awful, but it was surprisingly delightful!",
    "A masterpiece of sheer boredom and predictable cliches.",
    "The acting was good, but the screenplay ruined everything."
]

edge_inputs = tokenizer(edge_cases, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    edge_outputs = model(**edge_inputs)
    edge_probs = F.softmax(edge_outputs.logits, dim=1)
    edge_preds = torch.argmax(edge_probs, dim=1)

print("=== Kết Quả Kiểm Thử Ngôn Ngữ Phức Hợp ===\n")
for sent, pred, prob in zip(edge_cases, edge_preds, edge_probs):
    label = model.config.id2label[pred.item()]
    score = prob[pred].item()
    print(f"Câu: \"{sent}\"")
    print(f"Dự đoán: {label} (Độ tin cậy: {score*100:.2f}%)\n")
